# Animal Recognition System — Project Syllabus
> Wildlife identification in constrained environments using dual-camera visible/IR image capture

**Term:** August 2026 →  &nbsp;|&nbsp; **Development site:** California, USA &nbsp;|&nbsp; **Deployment site:** Uganda

---

### Mission Statement

Build a biodiversity-monitoring camera system for heavily wooded, low-light wilderness that recognizes animals from **partial, occluded, motion-blurred imagery** by fusing simultaneous visible-light photographs with aligned infrared (IR) heatmaps. Develop and validate on North American species (deer, mountain lion, coyote, fox, bear), then domain-adapt to Ugandan wildlife (gorillas and other indigenous species).

### Source Documents

This syllabus consolidates:
1. The high-level project definition (Block 1 of `Project_Plan_2026_August.ipynb`)
2. The "Useful Q&A with Gemini" research thread (Blocks 3–8 of the same notebook)

---
## Part I — Adopted Architecture Decision

### The Recommendation We Are Incorporating

Of the recommendations in the Gemini Q&A, the one that best matches the choices already made in the project definition (Block 1) is:

> **A two-stage hybrid pipeline — MegaDetector (CNN/YOLO) as detection frontend → BioCLIP (ViT) as species classifier — with the IR heatmap injected via *Early Fusion* (a 4-channel RGB+IR patch-embedding layer), training only lightweight heads over a frozen pretrained backbone.**

### Why this is the best fit for Block 1's choices

| Block 1 Choice | Matching Gemini Recommendation | Rationale |
|---|---|---|
| Frontend filter = **MegaDetector v5** | "MegaDetector as a pre-processor for a downstream classification layer is the exact gold standard" | Pretrained on millions of camera-trap images; already robust to occlusion, low light, and motion blur — directly solves the *partial-image deficit* out of the box. |
| Classifier = **BioCLIP (ViT)** | BioCLIP = "Top Recommendation" among downstream ViTs | ViT-Base backbone with contrastive language-image pretraining on iNaturalist; covers **both** North American mammals and African primates — ideal for the CA → Uganda migration. |
| "**Modify ViT model to accept IR layer with RGB image data**" | **Strategy A: Early Fusion (Input Adaptation)** — expand the patch-embedding layer from 3 → 4 channels (R, G, B + IR); rated ⭐⭐⭐⭐ for BioCLIP | Simplest viable fusion; only the new input embedding + classification head are trained, so a few-hundred-image dataset does not overfit. BioCLIP can still map fused features to its text vocabulary ("gorilla", "coyote") with very little data. |
| Small starting dataset (**hundreds of images**) | "Freeze the backbone… your few hundred images only train a small fraction of parameters" | Pure ViTs trained/heavily fine-tuned on small data overfit immediately (no inductive bias). Frozen-backbone few-shot heads neutralize the data-scarcity wall. |
| Target hardware = **Raspberry Pi 5 8GB + Hailo-8 26T AI HAT** | Gemini's own hardware verdict: "deploy using a Raspberry Pi 5 paired with a Hailo accelerator — best balance of field durability, low power draw, and high-speed local processing" | The Q&A independently converged on the hardware already chosen in Block 1. YOLO-family MegaDetector compiles cleanly to a Hailo `.hef`; 5–7 W power budget suits solar/battery field deployment. |

### Recorded fallback (do not implement yet)

If early-fusion accuracy stalls on heavily occluded subjects, the designated fallback is **Strategy B: Late Fusion** — dual frozen backbones (RGB stream + IR-as-grayscale stream) joined by a small trainable cross-attention bridge, with **DINOv2** (rated ⭐⭐⭐⭐⭐ for this strategy) replacing BioCLIP. This is deferred because it doubles backbone compute — a real cost on the Hailo-8 — and Block 1 explicitly specifies the single-stream RGB+IR-layer approach.

### End-to-end pipeline (adopted)

```
 Trigger (PIR / frame rate) 
        │
        ▼
 Dual capture: visible 1280×960  +  IR 256×192 (Thermal Master P4, 25 Hz)
        │
        ▼
 Preprocess & align: undistort → register IR to visible → shared image ID
        │
        ▼
 STAGE 1 — MegaDetector v5 (frozen)
   • rejects empty frames
   • returns bounding box [x_min, y_min, x_max, y_max]
        │
        ▼
 Context expansion: enlarge box 30–50%; crop BOTH visible and IR with same box
        │
        ▼
 STAGE 2 — BioCLIP ViT (frozen backbone)
   • 4-channel early-fusion patch embedding (trainable)
   • few-shot classification head (trainable)
        │
        ▼
 Species label + confidence  →  DUT results / field log
```

---
## Part II — Course Modules (Plan of Action)

Each module lists **objectives**, **tasks**, **deliverables**, and an **exit criterion** that gates the next module. Modules 1–2 can proceed in parallel.

---

### Module 0 — Project Setup & Literature Grounding *(Week 1)*

**Objectives:** reproducible dev environment; shared understanding of the two foundation models.

**Tasks**
1. Create the project repo; pin a Python environment (PyTorch, `megadetector` package, `open_clip`/BioCLIP weights, OpenCV, timm).
2. Run stock MegaDetector v5 on sample trail-cam images; confirm bounding-box JSON output.
3. Run stock BioCLIP zero-shot classification on sample RGB crops of the five target CA species.
4. Read the MegaDetector and BioCLIP model cards; note input resolutions, normalization constants, and license terms.

**Deliverables:** environment lockfile; two smoke-test notebooks (MegaDetector inference, BioCLIP zero-shot).

**Exit criterion:** both pretrained models produce sensible outputs on public sample images on the dev laptop.

---

### Module 1 — Capture Rig Bring-Up *(Weeks 1–3)*

**Objectives:** reliable, synchronized visible+IR capture from the Thermal Master P4.

**Tasks**
1. Bench-test the P4: verify the four dual-vision modes, 1280×960 visible stream, 256×192 VOx IR stream, 25 Hz frame rate, and −20…600 °C range.
2. Decide and document the IR data format (raw temperature array vs. pre-rendered heatmap) — **prefer raw radiometric data**, rendering to a normalized single channel in software so the fusion layer sees consistent input.
3. Build the capture script: timestamp-synchronized visible/IR frame pairs written with a shared image ID.
4. Characterize parallax between the two lenses at typical subject distances (5–30 m).

**Deliverables:** capture utility; parallax/`sync` characterization report; ≥100 test frame pairs (any subject).

**Exit criterion:** frame pairs are captured with < 1-frame timing skew and a documented, repeatable geometric relationship.

---

### Module 2 — Preprocessing & Alignment Pipeline *(Weeks 2–4)*

**Objectives:** implement Block 1's preprocessing steps as a reusable library.

**Tasks**
1. Calibrate both cameras (checkerboard for visible; heated calibration target for IR); compute the homography that maps the 256×192 IR frame onto visible coordinates.
2. Implement alignment: undistort → warp IR to visible frame → upsample IR to crop resolution (bilinear; preserve relative temperature values).
3. Implement the fixed-border animal isolation crop and the 30–50% context expansion around MegaDetector boxes, applied identically to both modalities.
4. Emit paired, aligned, ID-tagged crops in a documented dataset layout.

**Deliverables:** `preprocess/` library with unit tests; alignment error report (target: mean registration error ≤ ~2 visible-pixels at working distance).

**Exit criterion:** given a raw frame pair, the pipeline emits an aligned 4-channel (RGB+IR) crop with one command.

---

### Module 3 — Dataset Collection & Tagging *(Weeks 3–8, ongoing)*

**Objectives:** several hundred tagged, paired crops of CA wildlife.

**Tasks**
1. Field-deploy the rig in wooded CA sites; collect deer, coyote, fox, mountain lion, and bear encounters (expect deer to dominate — plan for class imbalance).
2. Run MegaDetector over all captures to pre-filter blanks; manually verify boxes and tag species (single shared ID per visible/IR pair, per Block 1).
3. Split data by *encounter/site*, not by frame, into train/val/test (~70/15/15) to prevent near-duplicate leakage.
4. Log per-image conditions (light level, occlusion severity, motion blur) to enable stratified evaluation later.

**Deliverables:** versioned dataset v1 (target ≥ 300–500 tagged pairs); labeling guide; class-balance report.

**Exit criterion:** dataset v1 frozen with clean splits and ≥ 5 examples per species in every split (fall back to merged "rare-carnivore" classes if bears/lions are too sparse).

---

### Module 4 — Stage-1 Frontend Integration *(Weeks 5–6)*

**Objectives:** MegaDetector wired in as the production frame filter and cropper.

**Tasks**
1. Wrap MegaDetector v5 inference behind a stable interface (`detect(frame) -> boxes, confidences`).
2. Tune the confidence threshold on val data for the *frame-filter* task: maximize blank rejection while keeping animal-frame recall ≥ 95%.
3. Implement the dual-modality crop step (same expanded box applied to RGB and aligned IR).
4. Evaluate frontend-only performance on the occlusion-stratified val set; this is the baseline the fusion classifier must beat.

**Deliverables:** frontend service module; threshold-tuning report; baseline detection metrics.

**Exit criterion:** ≥ 95% of animal-containing val frames pass the filter with a usable box.

---

### Module 5 — Stage-2 Classifier: BioCLIP + Early Fusion *(Weeks 6–10)* ⟵ **core of the project**

**Objectives:** the adopted recommendation — 4-channel early-fusion BioCLIP with frozen backbone.

**Tasks**
1. **Zero-shot baseline (RGB-only):** run frozen BioCLIP zero-shot on val crops. Record accuracy — this is the floor.
2. **Linear-probe baseline (RGB-only):** train only a classification head on RGB crops. This isolates the marginal value of IR later.
3. **Early-fusion surgery:** expand BioCLIP's patch-embedding weight from 3 → 4 input channels. Initialize the RGB channels from pretrained weights; initialize the IR channel randomly (small variance) or as the mean of the RGB channels. Freeze everything else.
4. **Train** the patch-embedding layer + classification head on dataset v1 with the targeted augmentation recipe (Module 6). Use a low LR (~1e-4), early stopping on val loss.
5. **(Stretch) contrastive alignment:** add an InfoNCE auxiliary loss pulling paired RGB/IR embeddings together, pushing unmatched pairs apart — per the Q&A, this keeps dark/blurry visible crops anchored to their clean thermal signatures.
6. Compare: zero-shot vs. RGB linear probe vs. RGB+IR early fusion, stratified by light level and occlusion severity. **The IR fusion must show its win specifically on the low-light/high-occlusion strata.**

**Deliverables:** training code; ablation table (3 configurations × condition strata); model checkpoint v1.

**Exit criterion:** early-fusion model beats the RGB-only linear probe on overall val accuracy **and** on the low-light stratum. If it does not, escalate to the recorded fallback (DINOv2 late-fusion cross-attention) with a documented decision memo.

---

### Module 6 — Targeted Augmentation & Training Optimization *(Weeks 7–10, interleaved with Module 5)*

**Objectives:** simulate the field's failure modes during training (per the Q&A's optimization recipe).

**Tasks**
1. **Random occlusion** (Cutout/Random Erasing, incl. bark/foliage-textured patches) — forces reliance on partial shapes.
2. **Directional motion blur** on the visible channel — mimics animals in motion.
3. **Low-light/contrast jitter** — severely degrade RGB brightness/contrast **while keeping the IR channel clean**, teaching the model to lean on IR exactly when vision fails.
4. Standard geometric augments (flip, small rotation, scale) applied *identically* to RGB and IR to preserve alignment.
5. Address class imbalance: weighted sampling or focal loss for rare carnivores.
6. **(Stretch, only if data grows to thousands)** multi-task auxiliary box/mask prediction head, per the Q&A's multi-task learning suggestion.

**Deliverables:** augmentation module with visual sanity-check notebook; before/after ablation rows added to the Module 5 table.

**Exit criterion:** augmentation demonstrably improves val accuracy on the degraded-condition strata without hurting clean-condition accuracy.

---

### Module 7 — Offline End-to-End System Test *(Weeks 10–12)*

**Objectives:** Block 1 step 7 — validate the full pipeline in offline mode on held-out data.

**Tasks**
1. Assemble the full chain (capture files → preprocess/align → MegaDetector → crop → fused BioCLIP → label) as one offline batch job.
2. Run on the untouched test split; report per-species precision/recall/F1, confusion matrix, and end-to-end latency per frame pair on the laptop.
3. Failure review: manually inspect every misclassification; categorize (alignment error, detector miss, fusion failure, label noise).
4. Fix the top failure category; re-run once. Freeze **System v1**.

**Deliverables:** end-to-end evaluation report; failure-mode taxonomy; frozen System v1 tag.

**Exit criterion:** documented test-set metrics with no unexplained failure clusters; go/no-go decision for edge porting.

---

### Module 8 — Edge Deployment: Raspberry Pi 5 + Hailo-8 *(Weeks 12–16)*

**Objectives:** port System v1 to the target hardware from Block 1.

**Tasks**
1. Export MegaDetector (YOLOv5) to ONNX → compile to a Hailo `.hef` with the Hailo Dataflow Compiler; validate detection parity vs. the laptop (mAP delta budget ≤ 2 points).
2. Quantize (INT8) using a calibration set drawn from dataset v1 (must include low-light images so quantization ranges cover them).
3. Port the fused BioCLIP classifier: attempt Hailo compilation; if ViT ops are unsupported, run Stage 2 on the Pi 5 CPU (acceptable — Stage 2 fires only on detector hits) and record throughput.
4. Rebuild the capture + preprocess chain on the Pi (P4 camera interface, alignment warp).
5. Measure the full power profile: idle / PIR-wait / active-inference; extrapolate solar+battery runtime for the Uganda duty cycle.
6. Burn-in: multi-day unattended field run in CA; compare edge outputs against laptop outputs on the same frames.

**Deliverables:** Hailo build scripts; parity report (accuracy laptop vs. edge); power budget document; field burn-in log.

**Exit criterion:** edge system matches laptop accuracy within budget and meets the field power envelope.

---

### Module 9 — Domain Adaptation: California → Uganda *(future term)*

**Objectives:** the Q&A's Phase-2 migration plan.

**Tasks**
1. Keep the frozen MegaDetector backbone and the trained fusion embedding **unchanged**.
2. Swap only the final classification head; retrain on a few hundred Ugandan crops (gorillas first — exploit BioCLIP's existing primate coverage and zero-shot capability for a head start).
3. Re-validate alignment/calibration on the deployed rigs (temperature range and vegetation differ).
4. Repeat Module 7's offline evaluation on Ugandan data before live monitoring.

**Deliverables:** Uganda classification head; adaptation evaluation report.

**Exit criterion:** gorilla/indigenous-species accuracy comparable to CA-phase results with ≤ a few hundred new labels.

---
## Part III — Milestone Schedule & Assessment

### Milestone summary

| # | Milestone | Target week | Gate |
|---|---|---|---|
| M0 | Both pretrained models running locally | 1 | Module 0 exit |
| M1 | Synchronized visible/IR capture proven | 3 | Module 1 exit |
| M2 | One-command aligned 4-channel crops | 4 | Module 2 exit |
| M3 | Dataset v1 frozen (≥300 tagged pairs) | 8 | Module 3 exit |
| M4 | Frontend filter ≥95% animal recall | 6 | Module 4 exit |
| M5 | **Fusion beats RGB-only baseline** | 10 | Module 5 exit — *primary project result* |
| M6 | System v1 end-to-end test report | 12 | Module 7 exit |
| M7 | Pi 5 + Hailo-8 parity & power budget | 16 | Module 8 exit |
| M8 | Uganda head trained & validated | future | Module 9 exit |

### Grading rubric (self-assessment)

- **Pass:** M0–M4 complete; RGB-only pipeline classifies CA species end-to-end.
- **Merit:** M5 — measurable IR-fusion gain on low-light/occluded strata (this is the project's novel claim).
- **Distinction:** M6–M7 — the full fused system running on the Pi 5/Hailo-8 within the field power budget.

### Standing risks & mitigations

| Risk | Likelihood | Mitigation |
|---|---|---|
| IR↔visible registration error swamps the fusion signal (256×192 vs 1280×960, parallax) | High | Module 2 error budget + calibration rig; evaluate fusion only after M2 gate passes. |
| Rare-species data starvation (lions, bears) | High | Merged rare classes; weighted loss; supplement with public camera-trap crops (RGB-only rows in a mixed-modality batch). |
| Early fusion underperforms | Medium | Pre-approved fallback: DINOv2 late-fusion cross-attention (Part I). |
| ViT unsupported by Hailo compiler | Medium | Stage 2 on Pi CPU is acceptable (event-driven, low duty cycle); revisit with Hailo SDK updates. |
| MegaDetector v6 availability/variants shift | Low | v5 is pinned for the term; evaluate v6 compact variants only at Module 8. |

### Open questions carried forward from the Q&A

1. Dual-camera geometry: is P4's fixed lens pair sufficient, or is a beam-splitter rig needed for tighter registration?
2. IR data format decision (raw radiometric array recommended) — confirm during Module 1.
3. Field power strategy in Uganda (solar vs. lithium packs) — needed before Module 8 power budgeting completes.
4. Frame-rate requirement: PIR-triggered burst vs. continuous 25 Hz — drives the edge duty cycle.

### Key references

- MegaDetector — https://github.com/agentmorris/MegaDetector
- BioCLIP — https://github.com/Imageomics/bioclip-2 · https://imageomics.github.io/bioclip/
- DINOv2 (fallback) — https://github.com/facebookresearch/dinov2
- DeepFaune (alternative camera-trap ViT) — https://www.deepfaune.cnrs.fr/en/
- Hailo-8 on Raspberry Pi 5 — https://www.jeffgeerling.com/blog/2026/frigate-with-hailo-for-object-detection-on-a-raspberry-pi/
- Full citation list: see the Q&A blocks in `Project_Plan_2026_August.ipynb`